# Milestone 4: Transformer Text Classification

This notebook replaces the Milestone 3 Keras embedding classifier with a transformer-based text classifier for review rating-band prediction.

It compares:

- **Baseline:** TF-IDF + Logistic Regression
- **Transformer:** DistilBERT (`distilbert-base-uncased`)
- **Fallback:** Frozen DistilBERT embeddings + Logistic Regression if local fine-tuning fails

Milestone 5 is not implemented here.

## Setup

`FAST_MODE` keeps runtime manageable on CPU by limiting row counts, sequence length, and epochs.

In [ ]:
from __future__ import annotations

import re
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid")
tf.keras.utils.set_random_seed(42)

FAST_MODE = True
RANDOM_SEED = 42
MODEL_NAME = "distilbert-base-uncased"
LABEL_ORDER = ["low", "medium", "high"]

if FAST_MODE:
    TRAIN_MAX_ROWS = 3000
    VALIDATION_MAX_ROWS = 1000
    TEST_MAX_ROWS = 1000
    EPOCHS = 1
    MAX_LENGTH = 128
else:
    TRAIN_MAX_ROWS = None
    VALIDATION_MAX_ROWS = None
    TEST_MAX_ROWS = None
    EPOCHS = 2
    MAX_LENGTH = 128

BATCH_SIZE = 16

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TRAIN_PATH = PROCESSED_DIR / "train.csv"
VALIDATION_PATH = PROCESSED_DIR / "validation.csv"
TEST_PATH = PROCESSED_DIR / "test.csv"

TEXT_CANDIDATES = ["review_text", "text", "review_body", "body", "content"]
RATING_CANDIDATES = ["review_rating", "rating", "average_rating", "overall", "score"]

print(f"Project root: {PROJECT_ROOT}")
print(f"FAST_MODE: {FAST_MODE}")
print(f"Model: {MODEL_NAME}")
print(f"Train/validation/test limits: {TRAIN_MAX_ROWS}, {VALIDATION_MAX_ROWS}, {TEST_MAX_ROWS}")
print(f"Epochs: {EPOCHS}; max_length: {MAX_LENGTH}")

## Load CSV Files

Load the processed train, validation, and test splits and print columns for each file.

In [ ]:
for path in [TRAIN_PATH, VALIDATION_PATH, TEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

train_raw = pd.read_csv(TRAIN_PATH, low_memory=False)
validation_raw = pd.read_csv(VALIDATION_PATH, low_memory=False)
test_raw = pd.read_csv(TEST_PATH, low_memory=False)

raw_splits = {
    "train": train_raw,
    "validation": validation_raw,
    "test": test_raw,
}

for name, df in raw_splits.items():
    print(f"{name} shape: {df.shape}")
    print(f"{name} columns: {df.columns.tolist()}")

## Prepare Text and Labels

The notebook detects the text and rating columns from known candidate names. It always recreates `rating_band` from the detected rating column.

In [ ]:
def detect_column(df: pd.DataFrame, candidates: list[str], split_name: str, kind: str) -> str:
    column = next((candidate for candidate in candidates if candidate in df.columns), None)
    if column is None:
        raise KeyError(
            f"No {kind} column found for {split_name}. "
            f"Expected one of {candidates}. Available columns: {df.columns.tolist()}"
        )
    return column


def clean_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and np.isnan(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def create_rating_band(rating_values: pd.Series) -> pd.Series:
    ratings = pd.to_numeric(rating_values, errors="coerce")
    bands = pd.Series(pd.NA, index=rating_values.index, dtype="string")
    bands.loc[ratings <= 2] = "low"
    bands.loc[(ratings > 2) & (ratings < 4)] = "medium"
    bands.loc[ratings >= 4] = "high"
    return bands


def limit_rows(df: pd.DataFrame, max_rows: int | None) -> pd.DataFrame:
    if max_rows is None or len(df) <= max_rows:
        return df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    return df.sample(n=max_rows, random_state=RANDOM_SEED).reset_index(drop=True)


def make_clean_split(df: pd.DataFrame, split_name: str, max_rows: int | None) -> pd.DataFrame:
    text_col = detect_column(df, TEXT_CANDIDATES, split_name, "text")
    rating_col = detect_column(df, RATING_CANDIDATES, split_name, "rating")
    print(f"{split_name}: text column = {text_col}; rating column = {rating_col}")

    clean_df = pd.DataFrame()
    clean_df["text"] = df[text_col].map(clean_text)
    clean_df["rating_band"] = create_rating_band(df[rating_col])
    clean_df = clean_df[(clean_df["text"].str.len() > 0) & clean_df["rating_band"].notna()].copy()
    clean_df["rating_band"] = clean_df["rating_band"].astype(str)
    clean_df = clean_df[clean_df["rating_band"].isin(LABEL_ORDER)]
    clean_df = clean_df[["text", "rating_band"]]
    clean_df = limit_rows(clean_df, max_rows)
    return clean_df[["text", "rating_band"]]


train_clean = make_clean_split(train_raw, "train", TRAIN_MAX_ROWS)
validation_clean = make_clean_split(validation_raw, "validation", VALIDATION_MAX_ROWS)
test_clean = make_clean_split(test_raw, "test", TEST_MAX_ROWS)

rating_counts = pd.DataFrame(
    {
        "train": train_clean["rating_band"].value_counts(),
        "validation": validation_clean["rating_band"].value_counts(),
        "test": test_clean["rating_band"].value_counts(),
    }
).fillna(0).astype(int)
display(rating_counts)
display(train_clean.head())

## Encode Labels

Convert rating bands to integer IDs for model training and evaluation.

In [ ]:
observed_labels = [label for label in LABEL_ORDER if label in set(train_clean["rating_band"])]
if len(observed_labels) < 2:
    raise ValueError("Training split must contain at least two rating bands.")

label_to_id = {label: index for index, label in enumerate(observed_labels)}
id_to_label = {index: label for label, index in label_to_id.items()}

validation_clean = validation_clean[validation_clean["rating_band"].isin(label_to_id)].reset_index(drop=True)
test_clean = test_clean[test_clean["rating_band"].isin(label_to_id)].reset_index(drop=True)

X_train_text = train_clean["text"].astype(str).tolist()
X_validation_text = validation_clean["text"].astype(str).tolist()
X_test_text = test_clean["text"].astype(str).tolist()

y_train = train_clean["rating_band"].map(label_to_id).to_numpy(dtype="int32")
y_validation = validation_clean["rating_band"].map(label_to_id).to_numpy(dtype="int32")
y_test = test_clean["rating_band"].map(label_to_id).to_numpy(dtype="int32")

print("Labels:", observed_labels)

# Baseline: TF-IDF + Logistic Regression

This baseline is fast and provides the latency comparison point for the transformer model.

In [ ]:
tfidf_model = Pipeline(
    steps=[
        ("tfidf", TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1)),
    ]
)

baseline_train_start = time.perf_counter()
tfidf_model.fit(X_train_text, y_train)
baseline_train_seconds = time.perf_counter() - baseline_train_start

baseline_pred_start = time.perf_counter()
tfidf_pred = tfidf_model.predict(X_test_text)
baseline_predict_seconds = time.perf_counter() - baseline_pred_start

baseline_accuracy = accuracy_score(y_test, tfidf_pred)
baseline_macro_f1 = f1_score(y_test, tfidf_pred, average="macro", zero_division=0)

print(f"TF-IDF accuracy: {baseline_accuracy:.4f}")
print(f"TF-IDF macro-F1: {baseline_macro_f1:.4f}")
print(f"TF-IDF train seconds: {baseline_train_seconds:.2f}")
print(f"TF-IDF predict seconds: {baseline_predict_seconds:.2f}")

# Transformer: DistilBERT

The preferred path fine-tunes `distilbert-base-uncased` with TensorFlow. If that fails locally, the notebook falls back to frozen DistilBERT embeddings plus Logistic Regression.

In [ ]:
transformer_method = "not_run"
transformer_pred = None
transformer_train_seconds = np.nan
transformer_predict_seconds = np.nan
transformer_error = None
tokenizer = None

try:
    from transformers import AutoTokenizer, TFAutoModelForSequenceClassification

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenize_for_tf(texts: list[str]) -> dict[str, np.ndarray]:
        encoded = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="np",
        )
        return {key: value for key, value in encoded.items()}

    train_tokens = tokenize_for_tf(X_train_text)
    validation_tokens = tokenize_for_tf(X_validation_text)
    test_tokens = tokenize_for_tf(X_test_text)

    transformer_model = TFAutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(observed_labels),
    )
    transformer_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"],
    )

    transformer_train_start = time.perf_counter()
    transformer_model.fit(
        train_tokens,
        y_train,
        validation_data=(validation_tokens, y_validation),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1,
    )
    transformer_train_seconds = time.perf_counter() - transformer_train_start

    transformer_predict_start = time.perf_counter()
    transformer_outputs = transformer_model.predict(test_tokens, batch_size=BATCH_SIZE, verbose=0)
    transformer_predict_seconds = time.perf_counter() - transformer_predict_start
    transformer_pred = np.argmax(transformer_outputs.logits, axis=1)
    transformer_method = "DistilBERT fine-tuning"
except Exception as exc:
    transformer_error = exc
    print(f"TensorFlow DistilBERT training failed locally: {exc}")
    print("Falling back to frozen DistilBERT embeddings + Logistic Regression.")

## Fallback: Frozen Transformer Embeddings

This section runs only if the fine-tuning path failed. It uses DistilBERT as a frozen feature extractor and trains Logistic Regression on pooled hidden states.

In [ ]:
if transformer_pred is None:
    try:
        from transformers import AutoTokenizer, TFAutoModel

        tokenizer = tokenizer or AutoTokenizer.from_pretrained(MODEL_NAME)
        frozen_model = TFAutoModel.from_pretrained(MODEL_NAME)
        frozen_model.trainable = False

        def frozen_embeddings(texts: list[str], batch_size: int = 32) -> np.ndarray:
            vectors = []
            for start in range(0, len(texts), batch_size):
                batch = texts[start : start + batch_size]
                encoded = tokenizer(
                    batch,
                    truncation=True,
                    padding=True,
                    max_length=MAX_LENGTH,
                    return_tensors="np",
                )
                outputs = frozen_model(**encoded, training=False)
                hidden = outputs.last_hidden_state.numpy()
                attention_mask = encoded["attention_mask"][..., None]
                masked_hidden = hidden * attention_mask
                pooled = masked_hidden.sum(axis=1) / np.clip(attention_mask.sum(axis=1), a_min=1, a_max=None)
                vectors.append(pooled)
            return np.vstack(vectors)

        transformer_train_start = time.perf_counter()
        X_train_transformer = frozen_embeddings(X_train_text)
        X_test_transformer = frozen_embeddings(X_test_text)
        frozen_classifier = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED)
        frozen_classifier.fit(X_train_transformer, y_train)
        transformer_train_seconds = time.perf_counter() - transformer_train_start

        transformer_predict_start = time.perf_counter()
        transformer_pred = frozen_classifier.predict(X_test_transformer)
        transformer_predict_seconds = time.perf_counter() - transformer_predict_start
        transformer_method = "Frozen DistilBERT embeddings + Logistic Regression"
    except Exception as fallback_exc:
        print(f"Frozen transformer fallback also failed: {fallback_exc}")
        print("Using TF-IDF predictions as an emergency fallback so later evaluation cells can run.")
        transformer_pred = tfidf_pred.copy()
        transformer_train_seconds = baseline_train_seconds
        transformer_predict_seconds = baseline_predict_seconds
        transformer_method = "Emergency TF-IDF fallback"

print("Transformer method:", transformer_method)

## Evaluation

Evaluate the transformer path on the test set and compare against the TF-IDF baseline.

In [ ]:
transformer_accuracy = accuracy_score(y_test, transformer_pred)
transformer_macro_f1 = f1_score(y_test, transformer_pred, average="macro", zero_division=0)

comparison = pd.DataFrame(
    [
        {
            "model": "TF-IDF Logistic Regression",
            "accuracy": baseline_accuracy,
            "macro_f1": baseline_macro_f1,
            "train_seconds": baseline_train_seconds,
            "predict_seconds": baseline_predict_seconds,
            "predict_ms_per_sample": baseline_predict_seconds / max(1, len(X_test_text)) * 1000,
        },
        {
            "model": transformer_method,
            "accuracy": transformer_accuracy,
            "macro_f1": transformer_macro_f1,
            "train_seconds": transformer_train_seconds,
            "predict_seconds": transformer_predict_seconds,
            "predict_ms_per_sample": transformer_predict_seconds / max(1, len(X_test_text)) * 1000,
        },
    ]
)
display(comparison)

print("Transformer classification report:")
print(
    classification_report(
        y_test,
        transformer_pred,
        labels=list(range(len(observed_labels))),
        target_names=observed_labels,
        zero_division=0,
    )
)

In [ ]:
def show_confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, title: str) -> None:
    matrix = confusion_matrix(y_true, y_pred, labels=list(range(len(observed_labels))))
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=observed_labels,
        yticklabels=observed_labels,
    )
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()


show_confusion_matrix(y_test, transformer_pred, "Transformer Test Confusion Matrix")

## Error Analysis

Inspect a small sample of transformer mistakes and disagreements with the TF-IDF baseline.

In [ ]:
error_df = pd.DataFrame(
    {
        "text": X_test_text,
        "actual": [id_to_label[int(label)] for label in y_test],
        "tfidf_pred": [id_to_label[int(label)] for label in tfidf_pred],
        "transformer_pred": [id_to_label[int(label)] for label in transformer_pred],
    }
)
error_df["transformer_correct"] = error_df["actual"] == error_df["transformer_pred"]
error_df["models_disagree"] = error_df["tfidf_pred"] != error_df["transformer_pred"]

print("Transformer error examples:")
display(error_df[~error_df["transformer_correct"]].head(10))

print("TF-IDF vs transformer disagreements:")
display(error_df[error_df["models_disagree"]].head(10))

## Complete

Milestone 4 is complete. Milestone 5 is not implemented.

In [ ]:
print("Milestone 4 Complete")